In [1]:
!pip install deep-translator --trusted-host pypi.org --trusted-host files.pythonhosted.org

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 3.0 MB/s eta 0:00:00


In [2]:
# baseline_zeroshot_translate_llm_embed.py

import json
import re
import time
from pathlib import Path

import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSeq2SeqLM


In [3]:
# ============================================================
# Settings
# ============================================================

DATA_DIR = Path("/kaggle/input/datasets/anasmhiri/aragenre/public_data_train_dev")
OUT_DIR = Path("baseline_outputs")
OUT_DIR.mkdir(exist_ok=True)

DEV_TEXT_FILE        = DATA_DIR / "dev.json"
DEV_GOLD_FILE        = DATA_DIR / "dev_gold.json"
DEV_DEFINITIONS_FILE = DATA_DIR / "dev_genre_definitions.json"

DETAILED_OUT_FILE    = OUT_DIR / "baseline_zeroshot_dev_predictions_detailed.json"
SUBMISSION_OUT_FILE  = OUT_DIR / "baseline_zeroshot_dev_submission.json"

TRANSLATION_MODEL      = "facebook/nllb-200-1.3B"
EMBEDDING_MODEL        = "sentence-transformers/sentence-t5-xxl"
QWEN_MODEL             = "Qwen/Qwen3-4B"

MAX_NEW_TOKENS         = 200
TEMPERATURE            = 0.0
TRANSLATION_BATCH_SIZE = 8
EMBEDDING_BATCH_SIZE   = 32

# Ensemble weights: raw text similarity vs Qwen prediction similarity
WEIGHT_TEXT = 0.4
WEIGHT_PRED = 0.6

RUN_LIMIT           = None
SLEEP_BETWEEN_CALLS = 0.0


In [4]:
# ============================================================
# Utilities
# ============================================================

def load_json(path):
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def save_json(path, data):
    with path.open("w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


def normalise(x):
    denom = np.linalg.norm(x, axis=1, keepdims=True)
    denom[denom == 0] = 1.0
    return x / denom


def cosine_matrix(a, b):
    return normalise(a) @ normalise(b).T


def evaluate(title, gold, pred):
    print(f"\n=== {title} ===")
    print("Accuracy:", round(accuracy_score(gold, pred), 4))
    print("Macro F1:", round(f1_score(gold, pred, average="macro", zero_division=0), 4))
    print(classification_report(gold, pred, digits=4, zero_division=0))


def hierarchical_macro_f1(gold_broad, pred_broad, gold_specific, pred_specific):
    """
    Official ranking metric for AraGenre.
    Average of Broad Macro F1 and Specific Macro F1.
    """
    broad_f1    = f1_score(gold_broad,    pred_broad,    average="macro", zero_division=0)
    specific_f1 = f1_score(gold_specific, pred_specific, average="macro", zero_division=0)
    hier_f1     = (broad_f1 + specific_f1) / 2.0

    print("\n=== Official Ranking Metric ===")
    print(f"Broad Macro F1:        {round(broad_f1,    4)}")
    print(f"Specific Macro F1:     {round(specific_f1, 4)}")
    print(f"Hierarchical Macro F1: {round(hier_f1,     4)}  ← official score")
    return hier_f1

In [5]:
# ============================================================
# Translation
# ============================================================

"""def load_translator():
    print("\nLoading NLLB translation model (cuda:0)...")
    tokenizer = AutoTokenizer.from_pretrained(TRANSLATION_MODEL)
    model = AutoModelForSeq2SeqLM.from_pretrained(
        TRANSLATION_MODEL,
        torch_dtype=torch.float16,
    )
    model = model.to("cuda:0")
    model.eval()
    return tokenizer, model


def translate_batch(texts, nllb_tokenizer, nllb_model):
    inputs = nllb_tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512,
    )
    inputs = {k: v.to("cuda:0") for k, v in inputs.items()}
    with torch.no_grad():
        outputs = nllb_model.generate(
            **inputs,
            forced_bos_token_id=nllb_tokenizer.convert_tokens_to_ids("eng_Latn"),
            max_length=512,
        )
    return [nllb_tokenizer.decode(o, skip_special_tokens=True) for o in outputs]"""

# Replace the load_translator() and translate_batch() functions with:

from deep_translator import GoogleTranslator

def load_translator():
    print("Using Google Translator...")
    return GoogleTranslator(source='ar', target='en')

def translate_batch(texts, translator):
    """Translate a batch of texts using Google Translator"""
    results = []
    for text in texts:
        try:
            translated = translator.translate(text)
            results.append(translated)
        except Exception as e:
            print(f"Translation error: {e}")
            results.append(text)  # fallback to original
    return results

def translate_all(texts, translator, desc="Translating"):
    """Translate all texts with progress bar"""
    results = []
    for i in tqdm(range(0, len(texts), TRANSLATION_BATCH_SIZE), desc=desc):
        batch = texts[i: i + TRANSLATION_BATCH_SIZE]
        results.extend(translate_batch(batch, translator))
    return results
    
"""def translate_all(texts, nllb_tokenizer, nllb_model, desc="Translating"):
    results = []
    for i in tqdm(range(0, len(texts), TRANSLATION_BATCH_SIZE), desc=desc):
        batch = texts[i: i + TRANSLATION_BATCH_SIZE]
        results.extend(translate_batch(batch, nllb_tokenizer, nllb_model))
    return results"""


def build_candidate_text_for_translation(c):
    return (
        f"Genre: {c['specific_genre']}. "
        f"Definition: {c['specific_genre_definition']}. "
        f"Broad genre: {c['broad_genre']}. "
        f"Broad genre definition: {c['broad_genre_definition']}."
    )

In [6]:
# ============================================================
# Qwen — zero-shot genre prediction
# ============================================================

def load_qwen():
    print("\nLoading Qwen tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(QWEN_MODEL)
    print("Loading Qwen model (cuda:1)...")
    model = AutoModelForCausalLM.from_pretrained(
        QWEN_MODEL,
        torch_dtype=torch.float16,
    )
    model = model.to("cuda:1")
    model.eval()
    return tokenizer, model


def build_prediction_prompt(text_en, candidates):
    """
    Show Qwen the candidate list as hints (not a hard constraint).
    The embedding snap step will correct any mismatch.
    Definitions are in English (already translated).
    """
    genre_hints = "\n".join([
        f"- {c['specific_genre']} ({c['broad_genre']}): {c['specific_genre_definition']}"
        for c in candidates
    ])
    return f"""You are a genre classification expert specialising in Arabic literature and media.

Known genre categories (use these as reference):
{genre_hints}

Read the text below carefully and describe its genre in detail.
Focus on: writing style, formality level, topic, register, and structural features.
The text may be Modern Standard Arabic, Classical Arabic, or a dialect — factor this into your reasoning.

Text:
{text_en}

Respond in this exact format:
Broad genre: <your answer>
Specific genre: <your answer>
Explanation: <detailed reasoning about style, register, and content>
"""


def call_qwen(prompt, qwen_tokenizer, qwen_model):
    messages = [
        {
            "role": "system",
            "content": (
                "You are an expert in Arabic literature, media, and genre classification. "
                "You are skilled at identifying genres across Modern Standard Arabic, "
                "Classical Arabic, and Arabic dialects."
            ),
        },
        {"role": "user", "content": prompt},
    ]
    input_text = qwen_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    )
    inputs = qwen_tokenizer(input_text, return_tensors="pt", truncation=True)
    inputs = {k: v.to("cuda:1") for k, v in inputs.items()}

    with torch.no_grad():
        outputs = qwen_model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            temperature=TEMPERATURE,
            pad_token_id=qwen_tokenizer.eos_token_id,
        )
    generated = qwen_tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    )
    return generated.strip()


def extract_explanation(qwen_output):
    """Extract only the Explanation section for a richer embedding signal."""
    match = re.search(r"Explanation:(.*)", qwen_output, re.DOTALL | re.IGNORECASE)
    if match:
        return match.group(1).strip()
    return qwen_output  # fallback: embed full output

def summarize_text(text_en, qwen_tokenizer, qwen_model):
    messages = [
        {"role": "user", "content": f"""Summarize the text delimited by triple backticks and only give the final answer. ```{text_en}```"""},
    ]
    input_text = qwen_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    )
    inputs = qwen_tokenizer(input_text, return_tensors="pt", truncation=True)
    inputs = {k: v.to("cuda:1") for k, v in inputs.items()}

    with torch.no_grad():
        outputs = qwen_model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            temperature=TEMPERATURE,
            pad_token_id=qwen_tokenizer.eos_token_id,
        )
    generated = qwen_tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    )
    return generated.strip()

In [7]:
# ============================================================
# Embedding & cosine matching
# ============================================================

def load_embedder():
    print("\nLoading sentence-t5-xxl embedder (cuda:0)...")
    return SentenceTransformer(EMBEDDING_MODEL, device="cuda:0")


def embed_texts(embedder, texts, desc="Embedding"):
    return embedder.encode(
        texts,
        convert_to_numpy=True,
        show_progress_bar=True,
        batch_size=EMBEDDING_BATCH_SIZE,
    )


def snap_to_candidate(text_emb, pred_emb, candidate_embs, candidates):
    """
    Ensemble: weighted combination of
      - direct text embedding vs candidates  (captures literal content)
      - Qwen explanation embedding vs candidates (captures reasoned genre signal)
    """
    sims_text = cosine_matrix(text_emb.reshape(1, -1), candidate_embs)[0]
    sims_pred = cosine_matrix(pred_emb.reshape(1, -1), candidate_embs)[0]
    final_sims = WEIGHT_TEXT * sims_text + WEIGHT_PRED * sims_pred
    best_idx = int(final_sims.argmax())
    return candidates[best_idx], float(final_sims[best_idx])

In [8]:
# ============================================================
# Main
# ============================================================

def main():
    # --------------------------------------------------------
    # Load data
    # --------------------------------------------------------
    print("Loading data...")
    dev_texts  = load_json(DEV_TEXT_FILE)
    dev_gold   = load_json(DEV_GOLD_FILE)
    candidates = load_json(DEV_DEFINITIONS_FILE)

    gold_by_id = {row["id"]: row for row in dev_gold}

    if RUN_LIMIT is not None:
        dev_texts = dev_texts[:RUN_LIMIT]

    print(f"Dev examples : {len(dev_texts)}")
    print(f"Candidates   : {len(candidates)}")
    print("\nCandidate genres:")
    for c in candidates:
        print(f"  {c['specific_genre']} ({c['broad_genre']})")

    """# --------------------------------------------------------
    # Step 1: Translate Arabic → English (cuda:0)
    # --------------------------------------------------------
    nllb_tokenizer, nllb_model = load_translator()

    print("\nTranslating candidate definitions...")
    candidate_arabic = [build_candidate_text_for_translation(c) for c in candidates]
    candidates_en = translate_all(
        candidate_arabic, nllb_tokenizer, nllb_model, "Translating candidates"
    )

    # Attach English definitions to candidates for the prompt
    candidates_with_en = [
        {**c, "specific_genre_definition": candidates_en[i]}
        for i, c in enumerate(candidates)
    ]

    print("\nTranslating dev texts...")
    dev_arabic = [row["text"] for row in dev_texts]
    dev_en = translate_all(
        dev_arabic, nllb_tokenizer, nllb_model, "Translating dev texts"
    )

    del nllb_model, nllb_tokenizer
    torch.cuda.empty_cache()
    print("Translator freed from GPU.")"""
    # In the main() function, replace the translator section with:
    
    # --------------------------------------------------------
    # Step 1: Translate Arabic → English (using Google)
    # --------------------------------------------------------
    translator = load_translator()
    
    print("\nTranslating candidate definitions...")
    candidate_arabic = [build_candidate_text_for_translation(c) for c in candidates]
    candidates_en = translate_all(
        candidate_arabic, translator, "Translating candidates"
    )
    
    # Attach English definitions to candidates for the prompt
    candidates_with_en = [
        {**c, "specific_genre_definition": candidates_en[i]}
        for i, c in enumerate(candidates)
    ]
    
    print("\nTranslating dev texts...")
    dev_arabic = [row["text"] for row in dev_texts]
    dev_en = translate_all(
        dev_arabic, translator, "Translating dev texts"
    )
    
    print("Translation complete. No GPU needed!")

    # --------------------------------------------------------
    # Step 2: Load Qwen (cuda:1)
    # --------------------------------------------------------
    qwen_tokenizer, qwen_model = load_qwen()

    # --------------------------------------------------------
    # Step 3: Load embedder + embed candidates (cuda:0)
    # --------------------------------------------------------
    embedder = load_embedder()

    print("Embedding candidate definitions...")
    candidate_embs = embed_texts(embedder, candidates_en, "Embedding candidates")

    # --------------------------------------------------------
    # Step 4: Main loop
    # --------------------------------------------------------
    detailed_predictions = []
    submission_predictions = []

    print("\nClassifying dev examples...")

    for i, row in enumerate(tqdm(dev_texts, desc="Classifying dev")):
        text_en = dev_en[i]

        # Qwen: free-form genre prediction with candidate hints
        prompt = build_prediction_prompt(text_en, candidates_with_en)
        try:
            qwen_output = call_qwen(prompt, qwen_tokenizer, qwen_model)
            # text_summarized = summarize_text(text_en, qwen_tokenizer, qwen_model)
            # print(f"Original text:{text_en}\t Summarized text: {text_summarized}")
        except Exception as e:
            qwen_output = f"ERROR: {repr(e)}"

        # Embed the explanation section (richest signal)
        explanation = extract_explanation(qwen_output)
        pred_emb = embedder.encode(explanation, convert_to_numpy=True, show_progress_bar=False)

        # Embed the raw translated text
        text_emb = embedder.encode(text_en, convert_to_numpy=True, show_progress_bar=False)

        # Ensemble snap to nearest valid candidate
        chosen_candidate, similarity = snap_to_candidate(
            text_emb, pred_emb, candidate_embs, candidates
        )

        pred_broad    = chosen_candidate["broad_genre"]
        pred_specific = chosen_candidate["specific_genre"]

        gold = gold_by_id[row["id"]]

        detailed_predictions.append({
            "id": row["id"],
            "text_ar": row["text"],
            "text_en": text_en,

            "gold_broad_genre":      gold["broad_genre"],
            "predicted_broad_genre": pred_broad,

            "gold_specific_genre":      gold["specific_genre"],
            "predicted_specific_genre": pred_specific,

            "ensemble_similarity":  similarity,
            "extracted_explanation": explanation,
            "raw_qwen_output":      qwen_output,
        })

        submission_predictions.append({
            "id": row["id"],
            "broad_genre":    pred_broad,
            "specific_genre": pred_specific,
        })

        print(
            f"[{i+1}/{len(dev_texts)}] "
            f"Gold: {gold['specific_genre']} | "
            f"Pred: {pred_specific} | "
            f"Sim: {similarity:.4f}"
        )

        if SLEEP_BETWEEN_CALLS > 0:
            time.sleep(SLEEP_BETWEEN_CALLS)

    # --------------------------------------------------------
    # Step 5: Save & evaluate
    # --------------------------------------------------------
    save_json(DETAILED_OUT_FILE, detailed_predictions)
    save_json(SUBMISSION_OUT_FILE, submission_predictions)

    gold_specific = [p["gold_specific_genre"]      for p in detailed_predictions]
    pred_specific = [p["predicted_specific_genre"] for p in detailed_predictions]
    gold_broad    = [p["gold_broad_genre"]          for p in detailed_predictions]
    pred_broad    = [p["predicted_broad_genre"]     for p in detailed_predictions]

    evaluate("Specific Genre Prediction", gold_specific, pred_specific)
    evaluate("Broad Genre Prediction",    gold_broad,    pred_broad)

    # Official ranking metric
    hierarchical_macro_f1(gold_broad, pred_broad, gold_specific, pred_specific)

    print("\nSaved:")
    print(DETAILED_OUT_FILE)
    print(SUBMISSION_OUT_FILE)


if __name__ == "__main__":
    main()

Loading data...
Dev examples : 110
Candidates   : 6

Candidate genres:
  Motivational Writing (Creative)
  Analytical Reports (Informative)
  Advice Columns (Interactive)
  Educational Explanations (Learning)
  Diplomatic Communication (Legal)
  Religious Commentary (Religious)
Using Google Translator...

Translating candidate definitions...


Translating candidates: 100%|██████████| 1/1 [00:00<00:00,  1.20it/s]



Translating dev texts...


Translating dev texts: 100%|██████████| 14/14 [00:15<00:00,  1.13s/it]

Translation complete. No GPU needed!

Loading Qwen tokenizer...


...

[3/110] Gold: Diplomatic Communication | Pred: Diplomatic Communication | Sim: 0.7891
Original text:The international mission welcomed the results of the recent negotiations between the two parties.	 Summarized text: <think>
Okay, the user wants me to summarize the given text and only provide the final answer. Let me look at the text again: "The international mission welcomed the results of the recent negotiations between the two parties." 

First, I need to identify the main points. The key elements here are the international mission, their action of welcoming, and the results of recent negotiations between two parties. The summary should be concise, so I should combine these elements without extra details.

I should check if there's any specific terminology or context that needs to be preserved. The term "international mission" is important because it indicates the entity involved. "Welcomed the results" is the action taken. The "recent negotiations between the two parties" is the su

Classifying dev:   4%|▎         | 4/110 [01:44<46:18, 26.21s/it]

[4/110] Gold: Diplomatic Communication | Pred: Diplomatic Communication | Sim: 0.7959
Original text:The delegation expressed its appreciation for the efforts made to support political dialogue.	 Summarized text: <think>
Okay, the user wants me to summarize the given text and only provide the final answer. Let me look at the text again: "The delegation expressed its appreciation for the efforts made to support political dialogue." 

First, I need to identify the main points. The key elements here are the delegation, their appreciation, and the efforts related to political dialogue. The sentence is pretty straightforward. The main action is expressing appreciation, so the summary should capture that.

I should make sure to keep it concise. Maybe start with "The delegation appreciated..." and then mention the efforts supporting political dialogue. Let me check if there's any unnecessary information. The original sentence doesn't have any extra details, so the summary should be a direct re

Classifying dev:   5%|▍         | 5/110 [02:11<45:49, 26.18s/it]

[5/110] Gold: Diplomatic Communication | Pred: Diplomatic Communication | Sim: 0.8066
